# CleanSight: messy data to clear insights

Upload a messy CSV or Excel file. CleanSight checks the data quality, cleans it, and writes an insights report for a non-technical manager.

**How it works:** Gemini decides which tools to run. The tools are Pandas functions, so every number comes from Pandas, not from the AI. Every change is written to a cleaning log.


## Setup

1. Get a free API key from Google AI Studio (aistudio.google.com).
2. In Colab, open the key icon (Secrets) on the left,paste your key, and turn on Notebook access.
3. Run the cells in order, from top to bottom.

## Cell 1: libraries aur file upload

In [ ]:
import numpy as np
import pandas as pd

from google.colab import files
uploaded = files.upload()


FILE_NAME = list(uploaded.keys())[0]
df = pd.read_csv(FILE_NAME)
print("File:", FILE_NAME, "| rows, columns:", df.shape)
df.head()

## Cell 2: Tool 1, check_quality()

In [ ]:
def check_quality(df):
    report = {}

    report["rows"] = len(df)
    report["missing"] = df.isnull().sum().to_dict()
    report["duplicates"] = int(df.duplicated().sum())

    outliers = {}
    for col in df.select_dtypes("number").columns:
        if df[col].nunique() <= 2:
            continue
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        low = q1 - 1.5 * iqr
        high = q3 + 1.5 * iqr
        outliers[col] = int(((df[col] < low) | (df[col] > high)).sum())
    report["outliers"] = outliers

    return report


report = check_quality(df)
for key, value in report.items():
    print(key, ":", value)

## Cell 3: Tool 2, clean_data()


In [ ]:
def clean_data(df):
    log = []
    clean_df = df.copy()

    # FIX 1: poora khaali column hatao
    for col in list(clean_df.columns):
        if clean_df[col].isnull().all():
            clean_df = clean_df.drop(columns=col)
            log.append(f"{col}: poora khaali tha, column hata diya")

    dup_count = int(clean_df.duplicated().sum())
    if dup_count > 0:
        clean_df = clean_df.drop_duplicates().reset_index(drop=True)
        log.append(f"{dup_count} duplicate rows hatayi")

    for col in clean_df.columns:
        miss_count = int(clean_df[col].isnull().sum())
        if miss_count == 0:
            continue
        if pd.api.types.is_numeric_dtype(clean_df[col]):
            median = clean_df[col].median()
            clean_df[col] = clean_df[col].fillna(median)
            log.append(f"{col}: {miss_count} khaali values median ({median}) se bhari")
        else:
            clean_df[col] = clean_df[col].fillna("Unknown")
            log.append(f"{col}: {miss_count} khaali values 'Unknown' se bhari")

    # FIX 2: make different spellings of the same value consistent
    for col in clean_df.select_dtypes(exclude="number").columns:
        if col == "date":
            continue
        s = clean_df[col]
        stripped = s.str.strip()
        key = stripped.str.lower()
        best = stripped.groupby(key).agg(lambda x: x.value_counts().idxmax())
        new = key.map(best)
        changed = int((s != new).sum())
        if changed > 0:
            clean_df[col] = new
            log.append(f"{col}: {changed} values made consistent (spelling/spacing)")

    return clean_df, log


clean_df, log = clean_data(df)

print("Pehle:", df.shape, " Baad mein:", clean_df.shape)
for line in log:
    print("-", line)
print(clean_df.isnull().sum().sum(), "missing bache")
print(clean_df.duplicated().sum(), "duplicates bache")

clean_df.to_csv("cleaned_data.csv", index=False)

## Cell 4: Tool 3, get_kpis()

In [ ]:
def get_kpis(df):
    kpis = {}
    kpis["rows"] = len(df)

    num_cols = []
    for col in df.select_dtypes("number").columns:
        if df[col].nunique() == len(df):
            continue
        num_cols.append(col)

    summary = {}
    for col in num_cols:
        summary[col] = {
            "total": round(float(df[col].sum()), 2),
            "average": round(float(df[col].mean()), 2),
            "median": round(float(df[col].median()), 2),
            "min": round(float(df[col].min()), 2),
            "max": round(float(df[col].max()), 2),
        }
    kpis["numbers"] = summary

    groups = {}
    for col in df.select_dtypes(exclude="number").columns:
        if df[col].nunique() > 10:
            continue
        result = {}
        for name, part in df.groupby(col):
            result[name] = {
                "count": len(part),
                "average": part[num_cols].mean().round(2).to_dict(),
                "median": part[num_cols].median().round(2).to_dict(),
            }
        groups[col] = result
    kpis["groups"] = groups

    return kpis


kpis = get_kpis(clean_df)

print("rows :", kpis["rows"])
print("numbers:", list(kpis["numbers"].keys()))
print("groups:", list(kpis["groups"].keys()))

## Cell 5: Gemini ke liye tool wrappers (aur testing without Gemini)

In [ ]:
data = {"raw": df, "clean": None, "log": None}


def quality_tool():
    """Check the uploaded file for missing values, duplicate rows and outliers. Returns the counts."""
    return check_quality(data["raw"])


def clean_tool():
    """Clean the uploaded file: remove duplicate rows and fill missing values. Returns a log of every change."""
    clean_df, log = clean_data(data["raw"])
    data["clean"] = clean_df
    data["log"] = log
    return {"rows_before": len(data["raw"]), "rows_after": len(clean_df), "changes": log}


def kpi_tool():
    """Calculate key numbers from the cleaned file: totals, averages and group-wise averages. Run clean_tool first."""
    if data["clean"] is None:
        return {"error": "Pehle clean_tool chalao"}
    return get_kpis(data["clean"])


# TEST (Gemini ke bina): teeno tools sahi order mein chalao
print("1) kpi pehle:", kpi_tool())
print("2) quality:", quality_tool())
print("3) clean:", clean_tool())
print("4) kpi baad mein, groups:", list(kpi_tool()["groups"].keys()))

## Cell 6: Gemini setup

In [ ]:
!pip install -q -U google-genai
!pip install -q gradio

In [ ]:
from google.colab import userdata
from google import genai
from google.genai import types
import gradio as gr

MODEL = "gemini-3.5-flash"

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
print("Gemini connection ready")

## Cell 7: Prompt and run_app

In [ ]:
PROMPT = """You are a data quality assistant. The user has uploaded a data file.
First call quality_tool and clean_tool, then call kpi_tool, then write a short report with 4 sections:
1. Data quality problems found
2. What was cleaned
3. Key numbers and 3 insights
4. Things to be careful about (mention outliers that were not removed and how they affect the averages)
5. Describe differences as observations, not causes. If group sizes are small or the differences are small, say so.
Get every number from the tools. Never invent or calculate numbers yourself.
Do not add currency symbols or units that are not in the data.
If a column has outliers, compare groups using the median, not the average, and say so.
For columns that only contain 0 and 1, the average is the rate. Ignore their median."""


def run_app(file):
    if file is None:
        return "Please upload a file first.", None

    if file.endswith(".csv"):
        data["raw"] = pd.read_csv(file)
    else:
        data["raw"] = pd.read_excel(file)
    data["clean"] = None
    data["log"] = None

    chat = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            system_instruction=PROMPT,
            tools=[quality_tool, clean_tool, kpi_tool],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=6),
        ),
    )
    response = chat.send_message("Clean the file and write the insights report.")

    if data["clean"] is None:
        return response.text, None

    data["clean"].to_csv("cleaned_data.csv", index=False)
    return response.text, "cleaned_data.csv"

## Cell 8: Gradio app (When we click the gradio button the request goes to gemini api)

In [ ]:
with gr.Blocks(title="CleanSight") as demo:
    gr.Markdown("# CleanSight\n### From messy data to clear insights")
    file_box = gr.File(label="Upload your file", file_types=[".csv", ".xlsx"])
    button = gr.Button("Clean and Report")
    download_box = gr.File(label="Cleaned file")
    report_box = gr.Markdown()
    button.click(run_app, inputs=file_box, outputs=[report_box, download_box])

demo.launch(share=True, debug=True)